# Handwritten Character Recognition - Model Training

This notebook trains a CNN on the EMNIST Letters dataset for character recognition (A-Z).

**Dataset:**
- EMNIST Letters: 26 classes (uppercase A-Z)
- ~145,600 training samples, ~24,400 test samples
- 28x28 grayscale images

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout, Flatten,
    InputLayer, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

plt.style.use('dark_background')
sns.set_palette('husl')

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Configuration

In [ ]:
CONFIG = {
    'batch_size': 128,
    'epochs': 30,
    'validation_split': 0.2,
    'use_augmentation': True,
    'checkpoint_dir': '../backend/checkpoints',
}

# Character labels
CHAR_LABELS = [chr(ord('A') + i) for i in range(26)]
print(f'Character Classes: {CHAR_LABELS}')

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

## 3. Load EMNIST Letters Dataset

In [ ]:
print('Loading EMNIST Letters dataset...')
print('(This may take a few minutes on first run)')

ds_train, ds_test = tfds.load(
    'emnist/letters',
    split=['train', 'test'],
    as_supervised=True,
    with_info=False
)

# Convert to numpy
X_train_full = []
y_train_full = []
for image, label in tfds.as_numpy(ds_train):
    X_train_full.append(image)
    y_train_full.append(label)

X_test = []
y_test = []
for image, label in tfds.as_numpy(ds_test):
    X_test.append(image)
    y_test.append(label)

X_train_full = np.array(X_train_full)
y_train_full = np.array(y_train_full)
X_test = np.array(X_test)
y_test = np.array(y_test)

print(f'Training samples: {X_train_full.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')
print(f'Original shape: {X_train_full.shape}')

## 4. Preprocess EMNIST Data

EMNIST images need to be transposed to fix orientation.

In [ ]:
# Transpose to fix orientation (EMNIST quirk)
if len(X_train_full.shape) == 4:
    X_train_full = np.transpose(X_train_full[:, :, :, 0], (0, 2, 1))
    X_test = np.transpose(X_test[:, :, :, 0], (0, 2, 1))
elif len(X_train_full.shape) == 3:
    X_train_full = np.transpose(X_train_full, (0, 2, 1))
    X_test = np.transpose(X_test, (0, 2, 1))

# Normalize
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape for CNN
X_train_full = X_train_full.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# EMNIST labels are 1-26, convert to 0-25
y_train_full = y_train_full - 1
y_test = y_test - 1

print(f'Processed shape: {X_train_full.shape}')
print(f'Label range: {y_train_full.min()} - {y_train_full.max()}')

In [ ]:
# One-hot encode and split
y_train_full_cat = to_categorical(y_train_full, 26)
y_test_cat = to_categorical(y_test, 26)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full_cat,
    test_size=CONFIG['validation_split'],
    random_state=42,
    stratify=y_train_full
)

print(f'Training: {X_train.shape[0]}')
print(f'Validation: {X_val.shape[0]}')
print(f'Test: {X_test.shape[0]}')

## 5. Visualize Sample Characters

In [ ]:
fig, axes = plt.subplots(4, 7, figsize=(14, 8))
fig.suptitle('Sample EMNIST Characters (One per Class)', fontsize=14)

y_train_labels = np.argmax(y_train, axis=1)

for i, ax in enumerate(axes.flat):
    if i < 26:
        idx = np.where(y_train_labels == i)[0][0]
        ax.imshow(X_train[idx].squeeze(), cmap='gray')
        ax.set_title(f'{CHAR_LABELS[i]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
unique, counts = np.unique(np.argmax(y_train, axis=1), return_counts=True)

plt.figure(figsize=(12, 5))
plt.bar([CHAR_LABELS[i] for i in unique], counts, color=plt.cm.viridis(np.linspace(0, 0.8, 26)))
plt.xlabel('Character')
plt.ylabel('Count')
plt.title('Training Set Class Distribution')
plt.xticks(rotation=0)
plt.show()

## 6. Define Character CNN Model

In [ ]:
def create_char_cnn():
    """CNN for 26-class character recognition."""
    model = Sequential([
        InputLayer(input_shape=(28, 28, 1)),
        
        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Dropout(0.25),
        
        # Dense layers
        Flatten(),
        Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        Dense(26, activation='softmax')  # 26 classes
    ], name='char_cnn')
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

char_model = create_char_cnn()
char_model.summary()

## 7. Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    fill_mode='nearest'
)

datagen.fit(X_train)

## 8. Training Callbacks

In [ ]:
callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(CONFIG['checkpoint_dir'], 'char_cnn_best.keras'),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

## 9. Train Model

In [ ]:
print('Training Character CNN with Data Augmentation...')

history = char_model.fit(
    datagen.flow(X_train, y_train, batch_size=CONFIG['batch_size']),
    steps_per_epoch=len(X_train) // CONFIG['batch_size'],
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    callbacks=callbacks,
    verbose=1
)

## 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val', linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Character CNN - Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val', linewidth=2, linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Character CNN - Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Evaluation

In [ ]:
test_loss, test_acc = char_model.evaluate(X_test, y_test_cat, verbose=0)

print('=' * 50)
print('TEST RESULTS')
print('=' * 50)
print(f'Character CNN Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
# Confusion Matrix
y_pred = char_model.predict(X_test, verbose=0)
y_pred_labels = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred_labels)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CHAR_LABELS, yticklabels=CHAR_LABELS)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Character CNN Confusion Matrix')
plt.show()

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred_labels, target_names=CHAR_LABELS))

## 12. Sample Predictions

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
fig.suptitle('Sample Character Predictions', fontsize=14)

indices = np.random.choice(len(X_test), 20, replace=False)

for i, ax in enumerate(axes.flat):
    idx = indices[i]
    ax.imshow(X_test[idx].squeeze(), cmap='gray')
    
    pred = y_pred_labels[idx]
    true = y_test[idx]
    conf = y_pred[idx, pred] * 100
    
    color = 'lime' if pred == true else 'red'
    ax.set_title(f'T:{CHAR_LABELS[true]} P:{CHAR_LABELS[pred]} ({conf:.1f}%)', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 13. Save Model

In [ ]:
char_model.save(os.path.join(CONFIG['checkpoint_dir'], 'char_cnn_final.keras'))
print(f"Model saved to {CONFIG['checkpoint_dir']}/char_cnn_final.keras")

## Summary

- Trained CNN on EMNIST Letters (26 classes: A-Z)
- Used data augmentation for better generalization
- Model saved for inference API
- Note: EMNIST images need transpose for correct orientation